# Análisis de Sentimiento, Emoción y NER — Boric vs Kast

Compara la cobertura de prensa de los primeros días del mandato de Gabriel Boric (2022) y José Antonio Kast (2026).
Mismo período del calendario (11 marzo al 8 mayo).

## 1. Imports

In [1]:
import glob
import re
import html
from collections import Counter

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

tqdm.pandas()
pd.set_option('display.max_colwidth', 120)

c:\Users\clien\Desktop\Comparativa-Gobiernos-Kast-Boric\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Carga del dataset

In [4]:
# Carga el CSV de gobierno (Boric + Kast)
df = pd.read_csv('resultados_gobierno2.csv', encoding='utf-8')

print(f'Dimensiones del dataset: {df.shape}')
print(f"Períodos: {df['ID recolección'].unique().tolist()}")
df.head(3)

Dimensiones del dataset: (2391, 10)
Períodos: ['boric 1', 'kast 1']


,Tipo,Texto,URL,Fecha Original,Fecha (dd/mm/yyyy),Fuente,Búsqueda Original,Etiqueta,ID recolección,Orientación política
0,Prensa,"<p>Tres puntos subió la aprobación del Presidente Gabriel Boric, de acuerdo a la última encuesta <a href=""https://ca...",https://www.biobiochile.cl/noticias/nacional/chile/2022/05/08/cadem-aprobacion-de-boric-sube-por-primera-vez-desde-q...,2022-05-08,08/05/2022,bbcl,presidente boric,bbcl,boric 1,Izquierda
1,Prensa,"<p>En el contexto de las demandas por faltas de recorridos, los gremios de conductores vienen alertando hace meses d...",https://www.biobiochile.cl/noticias/nacional/chile/2022/05/07/gobierno-capacitara-a-120-mujeres-para-que-sean-conduc...,2022-05-07,07/05/2022,bbcl,presidente boric,bbcl,boric 1,Izquierda
2,Prensa,"<p>El presidente de los trabajadores subcontratistas, Víctor Sepúlveda, señaló que no han tenido alguna señal o prop...",https://www.biobiochile.cl/noticias/nacional/region-del-bio-bio/2022/05/07/subcontratistas-aseguran-que-enap-evita-e...,2022-05-07,07/05/2022,bbcl,presidente boric,bbcl,boric 1,Izquierda


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2391 entries, 0 to 2390
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   Tipo                  2391 non-null   str  
 1   Texto                 2385 non-null   str  
 2   URL                   2391 non-null   str  
 3   Fecha Original        2391 non-null   str  
 4   Fecha (dd/mm/yyyy)    2391 non-null   str  
 5   Fuente                2391 non-null   str  
 6   Búsqueda Original     2391 non-null   str  
 7   Etiqueta              2391 non-null   str  
 8   ID recolección        2391 non-null   str  
 9   Orientación política  2391 non-null   str  
dtypes: str(10)
memory usage: 11.5 MB


In [6]:
TEXT_COL = 'Texto'
GROUP_COL = 'ID recolección'   # boric 1 / kast 1

print(f'Valores nulos en "{TEXT_COL}": {df[TEXT_COL].isna().sum()}')
print(f'\nDistribución por período:')
print(df[GROUP_COL].value_counts())

Valores nulos en "Texto": 6

Distribución por período:
ID recolección
kast 1     1773
boric 1     618
Name: count, dtype: int64


## 3. Limpieza del texto

In [7]:
def clean_text(raw: str) -> str:
    if not isinstance(raw, str):
        return ''

    text = html.unescape(raw)
    text = BeautifulSoup(text, 'lxml').get_text(separator=' ')
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'[@#]\w+', '', text)
    text = re.sub(r'[\x00-\x1f\x7f-\x9f]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def chunk_text(text: str, max_chars: int = 1500) -> list:
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks, current, current_len = [], [], 0
    for sent in sentences:
        if current and current_len + len(sent) + 1 > max_chars:
            chunks.append(' '.join(current))
            current, current_len = [sent], len(sent)
        else:
            current.append(sent)
            current_len += len(sent) + 1
    if current:
        chunks.append(' '.join(current))
    return chunks if chunks else [text[:max_chars]]


df['texto_limpio'] = df[TEXT_COL].progress_apply(clean_text)
df = df[df['texto_limpio'].str.len() > 0].reset_index(drop=True)

print(f'Filas tras limpieza: {len(df)}')
print('\nEjemplo limpio:')
print(df['texto_limpio'].iloc[0][:400])

100%|██████████| 2391/2391 [00:02<00:00, 1010.72it/s]

Filas tras limpieza: 2384

Ejemplo limpio:
Tres puntos subió la aprobación del Presidente Gabriel Boric, de acuerdo a la última encuesta Plaza Pública de Cadem difundida la noche de este domingo. Los guarismos recogidos entre el miércoles 4 y el viernes 6 de mayo, representan la primera mejora en la percepción de la labor del mandatario desde que asumió en La Moneda. Lee también... Boric y propuesta de revisar duración de su mandato: "Que 


## 4. Análisis de sentimiento

`pysentimiento` trunca a 512 tokens. `chunk_text` divide el artículo en fragmentos ≤ 1500 chars, se analiza cada uno y se promedian las probabilidades ponderadas por longitud de fragmento.

In [8]:
from pysentimiento import create_analyzer

sentiment_analyzer = create_analyzer(task='sentiment', lang='es')

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4194.66it/s]


In [9]:
def run_sentiment(text: str) -> dict:
    chunks = chunk_text(text)
    results = [sentiment_analyzer.predict(c).probas for c in chunks]
    weights = [len(c) for c in chunks]
    W = sum(weights)
    avg = {k: sum(r.get(k, 0) * w for r, w in zip(results, weights)) / W
           for k in ('POS', 'NEU', 'NEG')}
    label = max(avg, key=avg.get)
    return {
        'sent_label': label,
        'sent_pos': round(avg['POS'], 4),
        'sent_neu': round(avg['NEU'], 4),
        'sent_neg': round(avg['NEG'], 4),
    }

print('Ejecutando análisis de sentimiento...')
sent_results = df['texto_limpio'].progress_apply(run_sentiment)
df = pd.concat([df, pd.DataFrame(list(sent_results))], axis=1)
print('Listo.')
df[['texto_limpio', 'sent_label', 'sent_pos', 'sent_neu', 'sent_neg']].head(5)

Ejecutando análisis de sentimiento...


100%|██████████| 2384/2384 [09:05<00:00,  4.37it/s]

Listo.


,texto_limpio,sent_label,sent_pos,sent_neu,sent_neg
0,"Tres puntos subió la aprobación del Presidente Gabriel Boric, de acuerdo a la última encuesta Plaza Pública de Cadem...",NEU,0.0486,0.6094,0.3420
1,"En el contexto de las demandas por faltas de recorridos, los gremios de conductores vienen alertando hace meses de l...",NEG,0.0374,0.2709,0.6917
2,"El presidente de los trabajadores subcontratistas, Víctor Sepúlveda, señaló que no han tenido alguna señal o propues...",NEG,0.0120,0.0935,0.8945
3,"El general director de Carabineros, Ricardo Yáñez, realizó dos anuncios durante el funeral de Breant Rivas Manríquez...",NEG,0.0967,0.4409,0.4624
4,La Brigada de Homicidios de la Policía de Investigaciones indaga la causas que terminaron con un carabinero asesinad...,NEU,0.0844,0.4599,0.4557


## 5. Análisis de emociones

In [10]:
emotion_analyzer = create_analyzer(task='emotion', lang='es')
print('Analizador de emociones cargado.')

c:\Users\clien\Desktop\Comparativa-Gobiernos-Kast-Boric\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\clien\.cache\huggingface\hub\models--pysentimiento--robertuito-emotion-analysis. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 35221.22i

Analizador de emociones cargado.


In [11]:
EMO_KEYS = ('joy', 'anger', 'fear', 'sadness', 'disgust', 'surprise', 'others')

def run_emotion(text: str) -> dict:
    chunks = chunk_text(text)
    results = [emotion_analyzer.predict(c).probas for c in chunks]
    weights = [len(c) for c in chunks]
    W = sum(weights)
    avg = {k: sum(r.get(k, 0) * w for r, w in zip(results, weights)) / W
           for k in EMO_KEYS}
    label = max(avg, key=avg.get)
    return {'emo_label': label, **{f'emo_{k}': round(avg[k], 4) for k in EMO_KEYS}}

print('Ejecutando análisis de emociones...')
emo_results = df['texto_limpio'].progress_apply(run_emotion)
df = pd.concat([df, pd.DataFrame(list(emo_results))], axis=1)
print('Listo.')
df[['texto_limpio', 'emo_label'] + [f'emo_{k}' for k in EMO_KEYS]].head(5)

Ejecutando análisis de emociones...


100%|██████████| 2384/2384 [08:48<00:00,  4.51it/s]

Listo.


,texto_limpio,emo_label,emo_joy,emo_anger,emo_fear,emo_sadness,emo_disgust,emo_surprise,emo_others
0,"Tres puntos subió la aprobación del Presidente Gabriel Boric, de acuerdo a la última encuesta Plaza Pública de Cadem...",others,0.0020,0.0139,0.0018,0.0405,0.0029,0.0023,0.9367
1,"En el contexto de las demandas por faltas de recorridos, los gremios de conductores vienen alertando hace meses de l...",others,0.0007,0.0062,0.0028,0.0125,0.0009,0.0017,0.9752
2,"El presidente de los trabajadores subcontratistas, Víctor Sepúlveda, señaló que no han tenido alguna señal o propues...",anger,0.0009,0.8210,0.0013,0.0440,0.0280,0.0013,0.1035
3,"El general director de Carabineros, Ricardo Yáñez, realizó dos anuncios durante el funeral de Breant Rivas Manríquez...",others,0.0036,0.0239,0.0011,0.0921,0.0039,0.0021,0.8734
4,La Brigada de Homicidios de la Policía de Investigaciones indaga la causas que terminaron con un carabinero asesinad...,others,0.0038,0.0393,0.0076,0.0475,0.0106,0.0068,0.8845


## 6. Reconocimiento de entidades (NER)

In [12]:
ner_analyzer = create_analyzer(task='ner', lang='es')
print('Analizador NER cargado.')

# Verificar formato de salida con un ejemplo
test_ner = ner_analyzer.predict('Gabriel Boric es presidente de Chile y se reunió con el Banco Central.')
print('Ejemplo NER:', test_ner.entities[:3] if test_ner.entities else 'sin entidades')

c:\Users\clien\Desktop\Comparativa-Gobiernos-Kast-Boric\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\clien\.cache\huggingface\hub\models--pysentimiento--robertuito-ner. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 40007.02it/s]


Analizador NER cargado.
Ejemplo NER: [{'type': 'PER', 'text': 'Gabriel Boric', 'start': 0, 'end': 13}, {'type': 'LOC', 'text': 'Chile', 'start': 31, 'end': 36}, {'type': 'ORG', 'text': 'Banco Central', 'start': 56, 'end': 69}]


In [16]:
!python -m spacy download es_core_news_lg

     ---------------------------------------- 0.0/568.0 MB ? eta -:--:--
     --------------------------------------- 2.4/568.0 MB 20.8 MB/s eta 0:00:28
      ------------------------------------- 10.5/568.0 MB 35.6 MB/s eta 0:00:16
      ------------------------------------- 14.2/568.0 MB 30.6 MB/s eta 0:00:19
     - ------------------------------------ 21.0/568.0 MB 31.4 MB/s eta 0:00:18
     -- ----------------------------------- 31.5/568.0 MB 35.9 MB/s eta 0:00:15
     -- ----------------------------------- 41.9/568.0 MB 39.5 MB/s eta 0:00:14
     -- ----------------------------------- 41.9/568.0 MB 39.5 MB/s eta 0:00:14
     --- ---------------------------------- 52.4/568.0 MB 35.4 MB/s eta 0:00:15
     --- ---------------------------------- 55.1/568.0 MB 32.0 MB/s eta 0:00:17
     ---- --------------------------------- 60.8/568.0 MB 31.6 MB/s eta 0:00:17
     ---- --------------------------------- 64.0/568.0 MB 30.1 MB/s eta 0:00:17
     ---- --------------------------------- 67.

In [17]:
import spacy
nlp = spacy.load('es_core_news_lg')
print('Modelo spaCy:', nlp.meta['name'], '—', nlp.meta['version'])

def run_ner_spacy(text: str) -> dict:
    doc = nlp(text)
    per = [e.text.strip() for e in doc.ents if e.label_ == 'PER']
    org = [e.text.strip() for e in doc.ents if e.label_ == 'ORG']
    loc = [e.text.strip() for e in doc.ents if e.label_ in ('LOC', 'GPE')]
    return {
        'ner_personas': '|'.join(dict.fromkeys(per)),
        'ner_org':      '|'.join(dict.fromkeys(org)),
        'ner_lugar':    '|'.join(dict.fromkeys(loc)),
    }

print('Ejecutando NER con spaCy...')
ner_results = df['texto_limpio'].progress_apply(run_ner_spacy)

# Drop old empty NER cols if they exist, then concat
df.drop(columns=[c for c in ['ner_personas','ner_org','ner_lugar'] if c in df.columns], inplace=True)
df = pd.concat([df, pd.DataFrame(list(ner_results))], axis=1)
print('Listo.')
df[['texto_limpio', 'ner_personas', 'ner_org', 'ner_lugar']].head(5)


Modelo spaCy: core_news_lg — 3.8.0
Ejecutando NER con spaCy...


100%|██████████| 2384/2384 [03:47<00:00, 10.46it/s]

Listo.


,texto_limpio,ner_personas,ner_org,ner_lugar
0,"Tres puntos subió la aprobación del Presidente Gabriel Boric, de acuerdo a la última encuesta Plaza Pública de Cadem...",Presidente|Gabriel Boric|Lee|Boric|Jefe de Estado|Cadem Rechazo|Cadem|Cadem Violencia,Criteria|CC De|Rechazo|Convención|Carabineros,Plaza Pública de Cadem|La Moneda|Chillán|Santiago
1,"En el contexto de las demandas por faltas de recorridos, los gremios de conductores vienen alertando hace meses de l...",Susana Calderón|Ricardo Ruiz|Luis Cuello|Jeanette Jara,Servicio Nacional de Capacitación y Empleo|Partido Comunista|La Radio|Trabajo,Cerro Castillo
2,"El presidente de los trabajadores subcontratistas, Víctor Sepúlveda, señaló que no han tenido alguna señal o propues...",Víctor Sepúlveda|Lee|Sepúlveda|Monsalve|Subsecretario Monsalve|Boric,Empresa Nacional del Petróleo|Enap|Enap Sábado 07 Mayo,Sábado 07 Mayo
3,"El general director de Carabineros, Ricardo Yáñez, realizó dos anuncios durante el funeral de Breant Rivas Manríquez...",Ricardo Yáñez|Breant Rivas Manríquez|Yáñez|Lee|Boric|Breant Rivas|Vladimir Sáez | RBB|Rivas,Carabineros,Chillán|Ñuble|Renaico|La Araucanía|cuartel de Renaico|Yáñez
4,La Brigada de Homicidios de la Policía de Investigaciones indaga la causas que terminaron con un carabinero asesinad...,Gabriel Boric|Breant Rivas Manríquez|Sergio Pérez|Ricardo Yáñez|Breant Rivas|Izkia Siches,Brigada de Homicidios|Policía de Investigaciones|La Radio|Carabineros|Carabineros de Chile|Radio Bío Bío,Chillán|Catedral de Chillán|Ñuble|Prefectura de Carabineros|Interior|Renaico|La Araucanía


## Exp


In [15]:
# Términos de búsqueda por período
PRESIDENT_TERMS = {
    'boric 1': ['boric', 'gabriel boric', 'presidente'],
    'kast 1':  ['kast', 'josé antonio kast', 'jose antonio kast', 'presidente'],
}

def mentions_president(row):
    text = row['texto_limpio'].lower()
    terms = PRESIDENT_TERMS.get(row['ID recolección'], [])
    return any(t in text for t in terms)

df['menciona_presidente'] = df.apply(mentions_president, axis=1)

# Reporte
print('=== Artículos que mencionan al presidente correspondiente ===')
report = df.groupby('ID recolección')['menciona_presidente'].value_counts()
print(report)

print('\n=== % de cobertura por período ===')
print(
    df.groupby('ID recolección')['menciona_presidente']
    .mean().map('{:.1%}'.format)
)

# Vista de los que NO mencionan (para inspección)
no_match = df[~df['menciona_presidente']][['ID recolección', 'Fuente', 'texto_limpio']].head(10)
print(f'\nEjemplos sin mención ({(~df["menciona_presidente"]).sum()} en total):')
no_match


=== Artículos que mencionan al presidente correspondiente ===
ID recolección  menciona_presidente
boric 1         True                    610
                False                     8
kast 1          True                   1628
                False                   138
Name: count, dtype: int64

=== % de cobertura por período ===
ID recolección
boric 1    98.7%
kast 1     92.2%
Name: menciona_presidente, dtype: str

Ejemplos sin mención (146 en total):


,ID recolección,Fuente,texto_limpio
1,boric 1,bbcl,"En el contexto de las demandas por faltas de recorridos, los gremios de conductores vienen alertando hace meses de l..."
120,boric 1,bbcl,"Este 7 de abril, el Gobierno anunció el aumento del monto de la tarjeta de la Beca de Alimentación para la Educación..."
295,boric 1,emol,"En conversación con El Mercurio, el diputado independiente en la bancada PPD Jaime Araya se refirió a la tensión que..."
473,boric 1,emol,Las modificaciones legislativas propuestas para el período 2022-2026 1 Reforma tributaria. 2 Reformar la norma antie...
514,boric 1,emol,"Infografía: Johanna Mellado, Emol | Contenido: José Manuel Vilches, Alfonso González y Tomás Molina, Emol. | Fuentes..."
517,boric 1,emol,Infografía: Daniel Suárez E. | Contenido: Laura Gatica
518,boric 1,emol,Ministerio del Interior Manuel Monsalve Subsecretaría del Interior Edad: 56 años Fecha de nacimiento: 9 de julio de ...
570,boric 1,mediosregionales,"Una extensa jornada de trabajo sostuvo la Ministra Izkia Siches Pastén, con los y las dieciséis representantes regio..."
628,kast 1,bbcl,"En Más de 100 días revisamos las negociaciones del gobierno con el Partido de la Gente, sumado a las peleas internas..."
1029,kast 1,bbcl,"La nueva directiva de la Federación de Estudiantes (FEC) de la Universidad de Concepción, encabezada por Ivania Garr..."


In [18]:
df_export = df[df['menciona_presidente'] == True].copy()

output_cols = [
    'Tipo', 'Fecha (dd/mm/yyyy)', 'Fuente', 'Búsqueda Original',
    'ID recolección', 'Orientación política', 'texto_limpio',
    'sent_label', 'sent_pos', 'sent_neu', 'sent_neg',
    'emo_label', 'emo_joy', 'emo_anger', 'emo_fear',
    'emo_sadness', 'emo_disgust', 'emo_surprise', 'emo_others',
    'ner_personas', 'ner_org', 'ner_lugar',
]
output_cols = [c for c in output_cols if c in df.columns]

out_path = 'analisis_gob.csv'
df[output_cols].to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'Guardado: {out_path}  ({len(df)} filas, {len(output_cols)} columnas)')
print(df[output_cols].dtypes)


Guardado: analisis_gob.csv  (2384 filas, 22 columnas)
Tipo                        str
Fecha (dd/mm/yyyy)          str
Fuente                      str
Búsqueda Original           str
ID recolección              str
Orientación política        str
texto_limpio                str
sent_label                  str
sent_pos                float64
sent_neu                float64
sent_neg                float64
emo_label                   str
emo_joy                 float64
emo_anger               float64
emo_fear                float64
emo_sadness             float64
emo_disgust             float64
emo_surprise            float64
emo_others              float64
ner_personas                str
ner_org                     str
ner_lugar                   str
dtype: object


# Visualización 

## 7. Visualización comparativa Boric vs Kast

In [1]:
# Recargar resultados ya analizados (saltar secciones 4 y 5)
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('analisis_gob.csv', encoding='utf-8-sig')
df['fecha'] = pd.to_datetime(df['Fecha (dd/mm/yyyy)'], dayfirst=True, errors='coerce')
# Día relativo desde el inicio del mandato (11 marzo) — permite comparar 2022 vs 2026 en el mismo eje
df['mes_dia'] = df['fecha'].dt.strftime('%m-%d')
df['dia_mandato'] = (df['fecha'] - df.groupby('ID recolección')['fecha'].transform('min')).dt.days

GROUP_COL = 'ID recolección'
GROUPS = ['boric 1', 'kast 1']
GROUP_LABELS = {'boric 1': 'Boric (2022)', 'kast 1': 'Kast (2026)'}
GROUP_COLORS = {'Boric (2022)': '#1f77b4', 'Kast (2026)': '#d62728'}
df['Periodo'] = df[GROUP_COL].map(GROUP_LABELS)

print(f'{len(df)} filas cargadas.')
print(df['Periodo'].value_counts())

2384 filas cargadas.
Periodo
Kast (2026)     1766
Boric (2022)     618
Name: count, dtype: int64


In [2]:
# --- Gráfico 1: Distribución de sentimiento por período (barras apiladas %) ---
pivot_sent = (
    df.groupby(['Periodo', 'sent_label'])
    .size()
    .unstack(fill_value=0)
    .reindex(index=[GROUP_LABELS[g] for g in GROUPS])
    .reindex(columns=['POS', 'NEU', 'NEG'], fill_value=0)
)
pivot_pct = pivot_sent.div(pivot_sent.sum(axis=1), axis=0).mul(100).round(1).reset_index()
pivot_long = pivot_pct.melt(id_vars='Periodo', var_name='Sentimiento', value_name='Porcentaje')

label_map = {'POS': 'Positivo', 'NEU': 'Neutro', 'NEG': 'Negativo'}
pivot_long['Sentimiento'] = pivot_long['Sentimiento'].map(label_map)

fig1 = px.bar(
    pivot_long,
    x='Periodo', y='Porcentaje', color='Sentimiento',
    barmode='stack',
    color_discrete_map={'Positivo': '#4caf50', 'Neutro': '#9e9e9e', 'Negativo': '#f44336'},
    category_orders={'Sentimiento': ['Positivo', 'Neutro', 'Negativo']},
    text='Porcentaje',
    title='Distribución de sentimiento — Boric (2022) vs Kast (2026)',
)
fig1.update_traces(texttemplate='%{text:.0f}%', textposition='inside', textfont_size=12)
fig1.update_layout(
    xaxis_title='', yaxis_title='% de textos',
    yaxis_range=[0, 105],
    legend_title='Sentimiento',
    plot_bgcolor='white',
    height=450,
)
fig1.write_image('sentimiento_gobierno.png', scale=2)
fig1.show()

In [21]:
# --- Gráfico 2: Heatmap de probabilidades promedio de sentimiento ---
heat_data = (
    df.groupby('Periodo')[['sent_pos', 'sent_neu', 'sent_neg']]
    .mean()
    .reindex([GROUP_LABELS[g] for g in GROUPS])
    .rename(columns={'sent_pos': 'Positivo', 'sent_neu': 'Neutro', 'sent_neg': 'Negativo'})
    .round(3)
)

fig2 = px.imshow(
    heat_data,
    text_auto='.2f',
    color_continuous_scale='RdYlGn',
    zmin=0, zmax=0.7,
    aspect='auto',
    title='Probabilidad promedio de sentimiento por período',
)
fig2.update_layout(
    xaxis_title='', yaxis_title='',
    coloraxis_colorbar_title='Prob.',
    height=320,
)
fig2.write_image('heatmap_sentimiento_gobierno.png', scale=2)
fig2.show()

In [22]:
# --- Gráfico 4: Evolución diaria del sentimiento negativo (eje = día desde inauguración) ---
ts = (
    df.groupby(['Periodo', 'dia_mandato'])['sent_neg']
    .mean()
    .round(3)
    .reset_index()
)

fig4 = px.line(
    ts,
    x='dia_mandato', y='sent_neg', color='Periodo',
    markers=True,
    category_orders={'Periodo': [GROUP_LABELS[g] for g in GROUPS]},
    color_discrete_map=GROUP_COLORS,
    title='Evolución del sentimiento negativo por día de mandato',
    labels={'dia_mandato': 'Días desde el 11 de marzo', 'sent_neg': 'Prob. promedio NEG'},
)
fig4.update_layout(
    yaxis_range=[0, 1],
    plot_bgcolor='white',
    height=480,
    hovermode='x unified',
)
fig4.write_image('evolucion_sent_neg_gobierno.png', scale=2)
fig4.show()

In [23]:
# --- Gráfico 5: Volumen de cobertura por día y período ---
vol = (
    df.groupby(['Periodo', 'dia_mandato'])
    .size().reset_index(name='n')
)

fig5 = px.line(
    vol,
    x='dia_mandato', y='n', color='Periodo',
    markers=True,
    color_discrete_map=GROUP_COLORS,
    title='Volumen diario de cobertura — Boric vs Kast',
    labels={'dia_mandato': 'Días desde el 11 de marzo', 'n': 'N° de notas'},
)
fig5.update_layout(plot_bgcolor='white', height=420, hovermode='x unified')
fig5.write_image('volumen_gobierno.png', scale=2)
fig5.show()

In [24]:
# --- Gráfico 7: Distribución por fuente y período ---
fuente_pivot = (
    df.groupby(['Fuente', 'Periodo']).size()
    .unstack(fill_value=0)
    .reindex(columns=[GROUP_LABELS[g] for g in GROUPS], fill_value=0)
    .reset_index()
)
fuente_long = fuente_pivot.melt(id_vars='Fuente', var_name='Periodo', value_name='n')

fig7 = px.bar(
    fuente_long,
    x='Fuente', y='n', color='Periodo',
    barmode='group',
    color_discrete_map=GROUP_COLORS,
    text='n',
    title='Cobertura por fuente — Boric vs Kast',
)
fig7.update_traces(textposition='outside')
fig7.update_layout(plot_bgcolor='white', height=420, yaxis_title='N° de notas')
fig7.write_image('fuentes_gobierno.png', scale=2)
fig7.show()

In [4]:
# --- Gráfico: Distribución de emociones por período (barras apiladas %) ---
EMO_LABEL_MAP = {
    'joy': 'Alegría', 'anger': 'Enojo', 'fear': 'Miedo',
    'sadness': 'Tristeza', 'disgust': 'Asco', 'surprise': 'Sorpresa', 'others': 'Otra',
}
EMO_COLORS = {
    'Alegría': '#4caf50', 'Enojo': '#f44336', 'Miedo': '#9c27b0',
    'Tristeza': '#2196f3', 'Asco': '#795548', 'Sorpresa': '#ff9800', 'Otra': '#9e9e9e',
}

pivot_emo = (
    df.groupby(['Periodo', 'emo_label'])
    .size()
    .unstack(fill_value=0)
    .reindex(index=[GROUP_LABELS[g] for g in GROUPS])
)
pivot_emo_pct = pivot_emo.div(pivot_emo.sum(axis=1), axis=0).mul(100).round(1).reset_index()
pivot_emo_long = pivot_emo_pct.melt(id_vars='Periodo', var_name='Emoción', value_name='Porcentaje')
pivot_emo_long['Emoción'] = pivot_emo_long['Emoción'].map(EMO_LABEL_MAP).fillna(pivot_emo_long['Emoción'])

fig_emo1 = px.bar(
    pivot_emo_long,
    x='Periodo', y='Porcentaje', color='Emoción',
    barmode='stack',
    color_discrete_map=EMO_COLORS,
    text='Porcentaje',
    title='Distribución de emociones — Boric (2022) vs Kast (2026)',
)
fig_emo1.update_traces(texttemplate='%{text:.0f}%', textposition='inside', textfont_size=11)
fig_emo1.update_layout(
    xaxis_title='', yaxis_title='% de textos',
    yaxis_range=[0, 105],
    plot_bgcolor='white',
    height=480,
)
fig_emo1.write_image('emociones_gobierno.png', scale=2)
fig_emo1.show()

In [5]:
# --- Gráfico: Heatmap de probabilidades promedio de emoción ---
EMO_KEYS_VIZ = ('joy', 'anger', 'fear', 'sadness', 'disgust', 'surprise', 'others')
emo_cols_map = {f'emo_{k}': EMO_LABEL_MAP[k] for k in EMO_KEYS_VIZ}
heat_emo = (
    df.groupby('Periodo')[[f'emo_{k}' for k in EMO_KEYS_VIZ]]
    .mean()
    .reindex([GROUP_LABELS[g] for g in GROUPS])
    .rename(columns=emo_cols_map)
    .round(3)
)

fig_emo2 = px.imshow(
    heat_emo,
    text_auto='.3f',
    color_continuous_scale='YlOrRd',
    zmin=0, zmax=0.6,
    aspect='auto',
    title='Probabilidad promedio de emoción por período',
)
fig_emo2.update_layout(
    xaxis_title='', yaxis_title='',
    coloraxis_colorbar_title='Prob.',
    height=320,
)
fig_emo2.write_image('heatmap_emociones_gobierno.png', scale=2)
fig_emo2.show()

In [6]:
# --- Gráfico: Top 20 personas mencionadas por período ---
def top_entities(col, n=20):
    rows = []
    for _, row in df[['Periodo', col]].iterrows():
        for ent in str(row[col]).split('|'):
            ent = ent.strip()
            if ent:
                rows.append({'Periodo': row['Periodo'], 'Entidad': ent})
    counts = (
        pd.DataFrame(rows)
        .groupby(['Entidad', 'Periodo'])
        .size().reset_index(name='n')
    )
    top = counts.groupby('Entidad')['n'].sum().nlargest(n).index
    return counts[counts['Entidad'].isin(top)]

per_df = top_entities('ner_personas', 20)
fig_ner1 = px.bar(
    per_df.sort_values('n'),
    x='n', y='Entidad', color='Periodo',
    barmode='group',
    color_discrete_map=GROUP_COLORS,
    orientation='h',
    title='Top 20 personas más mencionadas — Boric vs Kast',
    labels={'n': 'N° de menciones', 'Entidad': ''},
)
fig_ner1.update_layout(plot_bgcolor='white', height=600, yaxis_categoryorder='total ascending')
fig_ner1.write_image('ner_personas_gobierno.png', scale=2)
fig_ner1.show()

In [7]:
# --- Gráfico: Top 15 organizaciones mencionadas por período ---
org_df = top_entities('ner_org', 15)
fig_ner2 = px.bar(
    org_df.sort_values('n'),
    x='n', y='Entidad', color='Periodo',
    barmode='group',
    color_discrete_map=GROUP_COLORS,
    orientation='h',
    title='Top 15 organizaciones más mencionadas — Boric vs Kast',
    labels={'n': 'N° de menciones', 'Entidad': ''},
)
fig_ner2.update_layout(plot_bgcolor='white', height=500, yaxis_categoryorder='total ascending')
fig_ner2.write_image('ner_org_gobierno.png', scale=2)
fig_ner2.show()